# Speaker diarization experiments, step by step (Colab)

This notebook reproduces the AMI experiments of the repository
[hothivijay/dpc6-speaker-diarization-experiments](https://github.com/hothivijay/dpc6-speaker-diarization-experiments):

| Step | What happens | Time on a T4 |
|---|---|---|
| 01 | download AMI headset-mix audio (170 meetings, ~10 GB) and the BUT references | 30–60 min |
| 02 | SA-EEND features (log-Mel, context, subsampling) | 10–20 min |
| 03 | train SA-EEND (Fujita et al., ASRU 2019) on AMI train | 40 min (time budget) |
| 04 | choose inference settings on **Dev only**, freeze them, decode Eval | 5–15 min |
| 05 | pyannote 3.1 baseline on AMI test (needs a Hugging Face token) | 10–20 min |
| 06 | score everything: DER under P1 / P2 / C25 with 95% CIs and paired tests | 1 min |
| 07 | edge check: ONNX FP32 vs INT8 on the CPU (speed, memory, DER change) | 10–20 min |
| 10 | collect all numbers into `results/summary.json` | 1 min |

Steps 08–09 (DISPLACE-2024) run only if you have the licensed DISPLACE data.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Keep this tab open on the free tier.
Every step can be re-run after a disconnect: finished work is skipped.

In [ ]:
# Is a GPU attached? (training on CPU would take days)
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"

## Get the code
Clones the repository to `/content/repo` and installs the few packages Colab does not already have.

In [ ]:
import os
if not os.path.isdir('/content/repo'):
    !git clone -q https://github.com/hothivijay/dpc6-speaker-diarization-experiments.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q -r requirements.txt

## Choose where the results are kept
* `USE_DRIVE = True` (recommended): features, models, system outputs and scores go to Google Drive
  (`MyDrive/dpc6_work`), so a disconnect loses nothing. The AMI audio stays on the fast local disk
  (`/content/ami`); after a restart, run step 01 again and it fetches only what is missing.
* `USE_DRIVE = False`: everything on the local disk (faster, but lost when the runtime is recycled).

In [ ]:
import os
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['WORK'] = '/content/drive/MyDrive/dpc6_work'
else:
    os.environ['WORK'] = '/content/work'
os.environ['AMI'] = '/content/ami'   # audio on the local disk (fast)
print('WORK =', os.environ['WORK'], '| AMI =', os.environ['AMI'])

## Optional: 1-minute smoke test
Runs the whole pipeline on tiny synthetic data, without downloading anything. The DER it prints means nothing;
it only shows that every step works. Its files go to `/tmp` and are deleted at the next run.

In [ ]:
!bash tests/smoke_test.sh 2>&1 | tail -n 12
!python -m pytest tests -q 2>&1 | tail -n 3

## Step 00: check the environment

In [ ]:
!bash steps/00_check_env.sh

## Step 01: download AMI
170 meetings of headset-mix audio from the official AMI mirror (CC BY 4.0) and the BUT `only_words`
references and UEMs (Full-corpus-ASR split: train 136, dev 18, test 16). If the cell stops, run it again.

In [ ]:
!bash steps/01_download_ami.sh 2>&1 | tail -n 15

## Step 02: features
23 log-Mel bands every 10 ms, ±7 frames of context, every 10th frame kept, giving one 345-dim vector per 100 ms.
The training targets mark which of the 4 most talkative speakers are active in each 100 ms frame.

In [ ]:
!bash steps/02_extract_features.sh 2>&1 | tail -n 5

## Step 03: train SA-EEND
4 Transformer blocks (256 units, 4 heads), no positional encoding, permutation-invariant BCE, Noam schedule.
`MAXMIN` is the time budget in minutes (default 40; the training time of the reference run is in `results/RESULTS.md`). If Colab disconnects, run the cell
again and training resumes from `last.pt`. At the end two averaged models are written: `avg.pt` (last 10 epochs,
as in the paper) and `avg_best.pt` (10 epochs around the lowest dev loss).

In [ ]:
!MAXMIN=40 bash steps/03_train_sa_eend.sh 2>&1 | grep -E "parameters|epoch|budget|averages"

## Step 04: choose settings on Dev, freeze them, decode Eval
Grid on the 14 Dev meetings: model average × window (whole, 500, 1500, 3000 frames) × threshold (0.3–0.7) ×
median filter (1, 5, 11). The best Dev setting is frozen and applied to Eval. Eval is never used for a choice.

In [ ]:
!bash steps/04_evaluate_sa_eend.sh 2>&1 | grep -E "best dev|FROZEN|eval|dev "

## Step 05: pyannote 3.1 baseline
1. Create a token at <https://huggingface.co/settings/tokens>.
2. Accept the conditions of `pyannote/speaker-diarization-3.1` and `pyannote/segmentation-3.0` on Hugging Face.
3. Add the token to Colab: 🔑 *Secrets* (left bar), name `HF_TOKEN`, then enable notebook access. Or paste it when asked.

pyannote.audio 3.3.2 needs older torch and numpy, so it gets its own environment in `/content/pyenv`.
Published value for this exact setting (model card, AMI headset mix, only_words, no collar, overlap scored): **18.8 %**.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token: ')
if not os.path.exists('/content/pyenv/bin/python'):
    !pip install -q uv && uv venv -q /content/pyenv --python 3.12 && uv pip install -q --python /content/pyenv/bin/python -r requirements-pyannote.txt
os.environ['PYANNOTE_PY'] = '/content/pyenv/bin/python'
!bash steps/05_pyannote_ami.sh 2>&1 | tail -n 5

## Step 06: score AMI Eval
DER under **P1** (no collar, overlap scored), P2 (0.25 s collar, overlap ignored) and C25 (0.25 s collar,
overlap scored), on 12 meetings (TNO excluded) and on all 16, with recording-bootstrap 95% CIs and the paired
SA-EEND − pyannote difference. See `docs/SCORING_PROTOCOLS.md`.

In [ ]:
!bash steps/06_score_ami.sh

## Step 07: edge check (ONNX FP32 vs INT8 on the CPU)
The frozen model is exported to ONNX and dynamically quantised to INT8. The whole pipeline (audio → features →
model → RTTM) then runs on the CPU with ONNX Runtime only (no PyTorch), on the 12 Eval meetings.
`docs/RASPBERRY_PI.md` shows how to run the same script on a Raspberry Pi 5.

In [ ]:
!bash steps/07_edge_benchmark.sh 2>&1 | grep -E "configuration|MB|rtf|ms_per_10s|peak_rss|DER"

## Steps 08–09: DISPLACE-2024 (optional, licensed data)
Only if you have the DISPLACE-2024 Track-1 eval data. Set the two folders and install NeMo first.

In [ ]:
# os.environ['DISPLACE_AUDIO'] = '/content/drive/MyDrive/DISPLACE/Track1_SD_supervised/Eval'   # your copy
# os.environ['DISPLACE_REF']   = '/content/drive/MyDrive/DISPLACE/eval_rttm'                    # your copy
# !pip install -q "nemo_toolkit[asr]"
# !bash steps/08_displace_systems.sh
# !bash steps/09_displace_rescore.sh

## Step 10: all numbers in one place

In [ ]:
!bash steps/10_collect_results.sh
print('summary file:', os.environ['WORK'] + '/results/summary.json')

### Everything at once
Instead of the cells above you can run all steps in order (finished steps are skipped):

In [ ]:
# !bash run_all.sh